In [2]:
import pandas as pd

# Read the data using the exact file name shown in your sidebar
df = pd.read_csv('Customer-Churn.csv')

# Display the first 5 rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# 1. See the total number of rows, columns, and data types
print("--- DATA INFO ---")
df.info()

print("\n--- MISSING VALUES ---")
# 2. Check if there are any missing values (blanks) in our data
df.isnull().sum()

--- DATA INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 n

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [4]:
df = df.drop('customerID', axis=1)

In [5]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

print("data cleaned! NEw data shape(Rows, columns):")
print(df.shape)

data cleaned! NEw data shape(Rows, columns):
(7032, 20)


In [8]:
print("--How many people left?--")
print(df['Churn'].value_counts())

df_encoded = pd.get_dummies(df, drop_first=True)

print("\n---Data ready for machine Learning---")
print(df_encoded.shape)

--How many people left?--
Churn
No     5163
Yes    1869
Name: count, dtype: int64

---Data ready for machine Learning---
(7032, 31)


In [10]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('Churn_Yes', axis=1)
y = df_encoded['Churn_Yes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("---Trainig Data(80%)---")
print(f"X_test shape: {X_test.shape}")

---Trainig Data(80%)---
X_test shape: (1407, 30)


In [12]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

print("training the model....please....wait")
model.fit(X_train, y_train)

print("Training complete! the model is ready")

training the model....please....wait
Training complete! the model is ready


In [18]:
from sklearn.metrics import accuracy_score, classification_report

prediction = model.predict(X_test)

accuracy = accuracy_score(y_test, prediction)
print(f"Overall Accuracy: {accuracy*100:.2f}%\n")

print("---detailed Report Card---")
print(classification_report(y_test, prediction))

Overall Accuracy: 78.54%

---detailed Report Card---
              precision    recall  f1-score   support

       False       0.83      0.90      0.86      1033
        True       0.63      0.48      0.54       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.70      1407
weighted avg       0.77      0.79      0.78      1407



In [19]:
import joblib

# 1. Save the trained model to a file
joblib.dump(model, 'churn_model.pkl')

# 2. Save the exact column names so our future web app knows what inputs to ask for
joblib.dump(list(X_train.columns), 'model_columns.pkl')

print("Model and columns saved successfully! Check your folder.")

Model and columns saved successfully! Check your folder.


In [20]:
import joblib
import pandas as pd

# 1. Load the saved "brain" and the column names
model = joblib.load('churn_model.pkl')
columns = joblib.load('model_columns.pkl')

# 2. Extract the importance scores
try:
    # For tree-based models (like Random Forest)
    scores = model.feature_importances_
except AttributeError:
    # For linear models (like Logistic Regression)
    scores = model.coef_[0]

# 3. Match the scores to the column names and display the top 10
importance_df = pd.DataFrame({'Feature': columns, 'Importance': scores})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("--- Top 10 Biggest Predictors of Churn ---")
print(importance_df.head(10))

--- Top 10 Biggest Predictors of Churn ---
                           Feature  Importance
3                     TotalCharges    0.193409
2                   MonthlyCharges    0.169758
1                           tenure    0.167572
10     InternetService_Fiber optic    0.039999
28  PaymentMethod_Electronic check    0.035016
13              OnlineSecurity_Yes    0.028905
25               Contract_Two year    0.028618
4                      gender_Male    0.026971
19                 TechSupport_Yes    0.025829
26            PaperlessBilling_Yes    0.025044
